# LPGNet — improved variants (A & B), hyperparameter search, and ablations

**Reference:** Alle & Priyakumar (2021), arXiv:2107.12878.
**Dataset:** PhysioNet `gaitpdb` 1.0.0 (same fixed 10-fold subject-level splits
as the replication notebook — same seed, same folds).

Everything here uses the **leakage-free fold-local protocol** (LP / predictor /
normalisation refit inside each fold on train-fold controls only).

| config | idea |
|---|---|
| **baseline** | LPGNet on fold-local LP residuals (= Exp 2 of the replication notebook, rerun here so this notebook is self-contained) |
| **Variant A** | SE-CNN with squeeze-excitation blocks + **attention pooling** on the fold-local LP residual |
| **Variant B** | **learned nonlinear predictor residual**: a small GRU next-step predictor (trained per fold on fold-local controls only) replaces the fixed order-11 LP; its residual feeds an LPGNet-style classifier |

**Protocol:** random hyperparameter search (nested-lite: inner 3-fold
subject-level CV *within the outer fold-0 training split*), then the best config
of each variant is evaluated on all 10 outer folds. Ablation: input type
{raw VGRF, LP residual, GRU residual} × inference mode {window-averaged,
whole-recording}.

*Runtime: this notebook is heavy (search = 2 × N_TRIALS × N_INNER small
trainings, plus ~60 more for final eval + ablation). On a Colab GPU expect
roughly 1–3 h with the default `N_TRIALS = 12`; reduce `N_TRIALS` for a quick
run.*


In [ ]:
# ============================================================
# Setup & global knobs
# ============================================================
import os, time, random, json, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import lfilter
from scipy.stats import wilcoxon
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---- knobs (edit freely) ----
N_FOLDS        = 10      # outer subject-level CV folds (paper: 10)
LP_ORDER       = 11      # linear-predictor order (paper: 11)
WIN, STRIDE    = 100, 50 # 2 s windows @50 Hz, 50% overlap (paper)
BATCH_WIN      = 128     # window-stage batch size (paper: 128)
LR_WIN         = 5e-4    # window-stage learning rate (paper: 5e-4)
LABEL_SMOOTH   = 0.1     # BCE label smoothing (paper: 0.1)
L2_REG         = 1e-4    # paper does NOT report the L2 value -> 1e-4 chosen (see notes)
MAX_EPOCHS_WIN = 60      # hard cap; early stopping usually fires much earlier
PATIENCE       = 8
VAL_FRAC       = 0.15    # fraction of TRAIN SUBJECTS held out for early stopping

print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ---- notebook-specific knobs ----
N_TRIALS    = 12   # random-search trials per variant (raise for a tighter search)
N_INNER     = 3    # inner subject-level CV folds used during search
SEARCH_FOLD = 0    # outer fold whose TRAIN split is used for the search
EPOCHS_PRED = 25   # max epochs for the GRU next-step predictor (early-stopped)
SEG         = 500  # predictor training segment length (samples @50 Hz)

In [ ]:
# ============================================================
# Download & parse PhysioNet gaitpdb
# (306 recordings / 166 subjects: 93 PD, 73 control; 18 VGRF channels, 100 Hz)
# ============================================================
URL = "https://physionet.org/static/published-projects/gaitpdb/gait-in-parkinsons-disease-1.0.0.zip"
ZIP = "gaitpdb.zip"; DATA_DIR = "gaitpdb_data"
if not os.path.exists(DATA_DIR):
    if not os.path.exists(ZIP):
        print("downloading ~300 MB (one-off) ...")
        urllib.request.urlretrieve(URL, ZIP)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DATA_DIR)
ROOT = os.path.join(DATA_DIR, "gait-in-parkinsons-disease-1.0.0")

records = []
for fn in sorted(os.listdir(ROOT)):
    if not fn.endswith(".txt") or fn[2:4] not in ("Co", "Pt"):
        continue
    base = fn[:-4]
    arr = np.loadtxt(os.path.join(ROOT, fn), delimiter="\t")
    sig = arr[:, 1:19].astype(np.float32)      # drop time column -> 18 VGRF channels
    records.append(dict(file=fn,
                        study=base[:2],                     # Ga / Ju / Si
                        subject=base.split("_")[0],         # e.g. GaPt15
                        label=0 if base[2:4] == "Co" else 1,  # 0=control, 1=PD
                        signal=sig))

subjects   = sorted({r["subject"] for r in records})
subj_label = {s: next(r["label"] for r in records if r["subject"] == s) for s in subjects}
n_pd  = sum(1 for s in subjects if subj_label[s] == 1)
n_co  = len(subjects) - n_pd
print(f"{len(records)} recordings | {len(subjects)} subjects ({n_pd} PD, {n_co} control)")
lens = np.array([len(r["signal"]) for r in records])
print(f"recording length @100 Hz: min {lens.min()}, median {int(np.median(lens))}, max {lens.max()}")

In [ ]:
# ============================================================
# Preprocessing (paper): causal 2-tap moving average, then decimate 100 -> 50 Hz
# Linear Prediction residuals: order 11, ONE predictor per channel, fit on
# concatenated CONTROL recordings with `order` zeros padding in between
# (this mirrors the paper's generateLPresidual.m exactly).
# ============================================================
def preprocess(x):
    return lfilter([0.5, 0.5], 1, x, axis=0)[::2]

def fit_lp(control_sigs, order=LP_ORDER):
    """Fit one order-`order` LP predictor per channel on concatenated controls.
    Returns (coeffs [18, order], stds [18]) -- stds = per-channel std of the
    concatenated control SIGNAL, used for normalisation, as in the paper.
    Autocorrelation via FFT (same values as the paper's xcorr, O(n log n))."""
    C, conc_all = np.zeros((18, order)), []
    for c in range(18):
        conc = np.concatenate([np.concatenate([s[:, c],
                                               np.zeros(order, np.float32)])
                               for s in control_sigs])
        conc_all.append(conc)
        n = len(conc)
        nfft = 1 << (2 * n - 1).bit_length()
        f = np.fft.rfft(conc, nfft)
        r = np.fft.irfft(f * np.conj(f), nfft)[:order + 1] / n   # biased autocorr
        R = np.array([[r[abs(i-j)] for j in range(order)] for i in range(order)])
        C[c] = np.linalg.solve(R, r[1:order+1])
    stds = np.std(np.stack(conc_all, axis=1), axis=0) + 1e-8
    return C, stds

def apply_lp(x, C, stds):
    """Prediction-error filter: e(n) = x(n) - sum_i c_i x(n-i), normalised."""
    e = np.stack([lfilter(np.r_[1.0, -C[c]], 1, x[:, c]) for c in range(18)], axis=1)
    return (e / stds).astype(np.float32)

def set_feats(idx, C, stds, key="feat"):
    """Compute LP-residual features for a set of recordings in-place."""
    for i in idx:
        records[i][key] = apply_lp(records[i]["proc"], C, stds)

for r in records:                       # preprocessed signals, computed once
    r["proc"] = preprocess(r["signal"])

In [ ]:
# ============================================================
# Fixed 10-fold STRATIFIED SUBJECT-LEVEL splits.
# Fixed seed -> every experiment in this notebook uses IDENTICAL folds,
# so configurations are directly comparable fold-by-fold.
# ============================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
subj_arr = np.array(subjects)
subj_y   = np.array([subj_label[s] for s in subjects])
FOLDS = []
for tr_s, te_s in skf.split(subj_arr, subj_y):
    tr_subj, te_subj = set(subj_arr[tr_s]), set(subj_arr[te_s])
    FOLDS.append(dict(
        train=[i for i, r in enumerate(records) if r["subject"] in tr_subj],
        test =[i for i, r in enumerate(records) if r["subject"] in te_subj]))
for k, f in enumerate(FOLDS):
    ntr = len({records[i]["subject"] for i in f["train"]})
    nte = len({records[i]["subject"] for i in f["test"]})
    print(f"fold {k}: {len(f['train'])} train recs ({ntr} subj) | "
          f"{len(f['test'])} test recs ({nte} subj)")

In [ ]:
# ============================================================
# LPGNet classifier (Fig. 2 of the paper)
#   SpatialDropout(0.2)
#   -> SepConv 7x32 -> BN -> ELU -> MaxPool
#   -> SepConv 5x32 -> BN -> ELU -> MaxPool
#   -> SepConv 3x32 -> BN -> ELU -> GlobalAveragePooling
#   -> Dropout(0.25) -> Dense(1, sigmoid)
# (MaxPool size is not reported; pool size 2 assumed.)
# ============================================================
def build_lpgnet(T, n_ch=18, channels=32, kernels=(7, 5, 3),
                 spat_drop=0.2, drop=0.25, l2=L2_REG):
    reg = keras.regularizers.l2(l2)
    # NOTE: variable-length input (None) -> the same network can process
    # 100-sample windows (step 1) AND whole recordings (step 2).
    inp = layers.Input((None, n_ch))
    x = layers.SpatialDropout1D(spat_drop)(inp)
    for i, k in enumerate(kernels):
        x = layers.SeparableConv1D(channels, k, padding="same",
                                   depthwise_regularizer=reg,
                                   pointwise_regularizer=reg)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("elu")(x)
        if i < len(kernels) - 1:
            x = layers.MaxPooling1D(2)(x)
    x = layers.GlobalAveragePooling1D(name="pooled")(x)
    x = layers.Dropout(drop)(x)
    out = layers.Dense(1, activation="sigmoid", kernel_regularizer=reg)(x)
    return keras.Model(inp, out)

_m = build_lpgnet(WIN)
print(f"LPGNet trainable params (window input): {_m.count_params():,}")
print("Paper reports 4,735 CNN params (4,933 incl. the 198 LP coefficients).")
print("Minor differences vs this Keras rebuild can come from unspecified")
print("details (pool size, padding, BN parameters, where L2 is applied).")
del _m

In [ ]:
# ============================================================
# Two-step training pipeline (paper, Sec. II-C)
#  step 1: train the whole network on 2 s windows (50% overlap)
#  step 2: freeze the conv backbone, recalibrate the FC layer on WHOLE
#          recordings. We implement step 2 exactly: run the frozen backbone
#          over each whole recording -> 32-d GAP embedding -> logistic
#          regression (mathematically the same objective as retraining
#          Dense(1,sigmoid) with cross-entropy, but deterministic).
# ============================================================
def make_windows(idx, key="feat"):
    Xw, yw = [], []
    for i in idx:
        s = records[i][key]
        for st in range(0, len(s) - WIN + 1, STRIDE):
            Xw.append(s[st:st + WIN]); yw.append(records[i]["label"])
    return np.asarray(Xw, np.float32), np.asarray(yw, np.float32)

def split_val(idx, frac=VAL_FRAC, seed=SEED):
    """Hold out a fraction of TRAIN SUBJECTS for early stopping
    (subject-level, stratified when both classes are present)."""
    ss = sorted({records[i]["subject"] for i in idx})
    yy = [subj_label[s] for s in ss]
    if len(set(yy)) > 1:
        sss = StratifiedShuffleSplit(1, test_size=frac, random_state=seed)
        tr_s, va_s = next(sss.split(np.zeros(len(ss)), yy))
    else:
        rng = np.random.default_rng(seed)
        perm = rng.permutation(len(ss))
        n_va = max(1, int(round(frac * len(ss))))
        va_s, tr_s = perm[:n_va], perm[n_va:]
    tr_subj = {ss[i] for i in tr_s}; va_subj = {ss[i] for i in va_s}
    return ([i for i in idx if records[i]["subject"] in tr_subj],
            [i for i in idx if records[i]["subject"] in va_subj])

def train_window_stage(build_fn, train_idx, seed=SEED, lr=LR_WIN, verbose=0,
                       key="feat"):
    tf.keras.utils.set_random_seed(seed)
    tr_i, va_i = split_val(train_idx)
    Xtr, ytr = make_windows(tr_i, key=key); Xva, yva = make_windows(va_i, key=key)
    model = build_fn(WIN)
    model.compile(optimizer=keras.optimizers.Adam(lr, clipnorm=1.0),
                  loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                  metrics=["accuracy"])
    cb = [keras.callbacks.EarlyStopping("val_loss", patience=PATIENCE,
                                        restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.25, patience=3)]
    model.fit(Xtr, ytr, validation_data=(Xva, yva), batch_size=BATCH_WIN,
              epochs=MAX_EPOCHS_WIN, callbacks=cb, verbose=verbose)
    return model

def embedding_model(model):
    """Frozen backbone up to the pooled representation."""
    return keras.Model(model.input, model.get_layer("pooled").output)

def recalibrate_fc(model, train_idx, key="feat"):
    """Step 2: frozen backbone -> embedding -> logistic regression."""
    emb = embedding_model(model)
    E = np.stack([emb.predict(records[i][key][None], verbose=0)[0] for i in train_idx])
    y = np.array([records[i]["label"] for i in train_idx])
    clf = LogisticRegression(max_iter=2000)
    clf.fit(E, y)
    return clf

def predict_recordings(model, clf, idx, key="feat"):
    """Whole-recording inference -> PD probability per recording."""
    emb = embedding_model(model)
    return np.asarray([clf.predict_proba(emb.predict(records[i][key][None],
                       verbose=0)[0][None])[0, 1] for i in idx])

def predict_window_avg(model, idx, key="feat"):
    """Alternative inference: average window probabilities over a recording."""
    out = []
    for i in idx:
        s = records[i][key]
        ws = np.stack([s[st:st + WIN] for st in range(0, len(s) - WIN + 1, STRIDE)])
        out.append(float(model.predict(ws, verbose=0).mean()))
    return np.asarray(out)

def compute_metrics(y_true, p):
    y_hat = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()
    return dict(AUC=roc_auc_score(y_true, p),
                ACC=accuracy_score(y_true, y_hat),
                F1=f1_score(y_true, y_hat),          # PD = positive class (paper)
                SENS=tp / (tp + fn), SPEC=tn / (tn + fp))

In [ ]:
# ============================================================
# Variant A: SE-CNN + attention pooling (input: fold-local LP residual)
# ============================================================
def se_block(x, ratio):
    ch = x.shape[-1]
    s = layers.GlobalAveragePooling1D()(x)
    s = layers.Dense(max(ch // ratio, 4), activation="relu")(s)
    s = layers.Dense(ch, activation="sigmoid")(s)
    return layers.Multiply()([x, layers.Reshape((1, ch))(s)])

def attn_pool(x):
    w = layers.Softmax(axis=1)(layers.Dense(1)(x))       # (T,1) attention weights
    return layers.Lambda(lambda t: tf.reduce_sum(t[0] * t[1], axis=1),
                         name="pooled")([x, w])

def build_variantA(T, n_ch=18, channels=32, kernels=(7, 5, 3), se_ratio=8,
                   spat_drop=0.2, drop=0.25, l2=L2_REG):
    reg = keras.regularizers.l2(l2)
    # variable-length input: windows (step 1) and whole recordings (step 2)
    inp = layers.Input((None, n_ch))
    x = layers.SpatialDropout1D(spat_drop)(inp)
    for i, k in enumerate(kernels):
        x = layers.SeparableConv1D(channels, k, padding="same",
                                   depthwise_regularizer=reg,
                                   pointwise_regularizer=reg)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("elu")(x)
        x = se_block(x, se_ratio)
        if i < len(kernels) - 1:
            x = layers.MaxPooling1D(2)(x)
    x = attn_pool(x)
    x = layers.Dropout(drop)(x)
    out = layers.Dense(1, activation="sigmoid", kernel_regularizer=reg)(x)
    return keras.Model(inp, out)

def _build_kw(cfg, *drop_keys):
    return {k: v for k, v in cfg.items() if k not in drop_keys}

def builder_for(variant, cfg):
    if variant == "A":
        return lambda T: build_variantA(T, **_build_kw(cfg, "lr"))
    if variant == "B":
        return lambda T: build_lpgnet(T, **_build_kw(cfg, "lr", "gru_hidden"))
    return build_lpgnet

_m = build_variantA(WIN)
print(f"Variant A (default cfg) trainable params: {_m.count_params():,}")
del _m

In [ ]:
# ============================================================
# Variant B: small GRU NEXT-STEP predictor -> residual -> LPGNet-style CNN.
# The predictor is trained PER FOLD on fold-local CONTROL recordings only
# (the exact analogue of the LP fit, so the protocol stays leakage-free).
# ============================================================
def build_predictor(n_ch=18, hidden=32):
    inp = layers.Input((None, n_ch))
    x = layers.GRU(hidden, return_sequences=True)(inp)
    out = layers.TimeDistributed(layers.Dense(n_ch))(x)
    return keras.Model(inp, out)

def pred_segments(idx, mu, sd):
    Xs, Ys = [], []
    for i in idx:
        z = ((records[i]["proc"] - mu) / sd).astype(np.float32)
        for st in range(0, len(z) - SEG, SEG):
            seg = z[st:st + SEG]
            Xs.append(seg[:-1]); Ys.append(seg[1:])
    return np.stack(Xs), np.stack(Ys)

def train_predictor(train_idx, hidden=32, epochs=EPOCHS_PRED, seed=SEED, verbose=0):
    """Next-step predictor trained ONLY on fold-local control recordings."""
    tf.keras.utils.set_random_seed(seed)
    ctr = [i for i in train_idx if records[i]["label"] == 0]
    tr_i, va_i = split_val(ctr, frac=0.15, seed=seed)
    cat = np.concatenate([records[i]["proc"] for i in tr_i])
    mu, sd = cat.mean(0), cat.std(0) + 1e-8
    Xtr, Ytr = pred_segments(tr_i, mu, sd)
    Xva, Yva = pred_segments(va_i, mu, sd)
    model = build_predictor(hidden=hidden)
    model.compile(optimizer=keras.optimizers.Adam(1e-3, clipnorm=1.0), loss="mse")
    model.fit(Xtr, Ytr, validation_data=(Xva, Yva), batch_size=32, epochs=epochs,
              callbacks=[keras.callbacks.EarlyStopping("val_loss", patience=4,
                                                      restore_best_weights=True)],
              verbose=verbose)
    return model, mu, sd

def predictor_residual(model, x, mu, sd, rstd):
    z = ((x - mu) / sd).astype(np.float32)
    pred = model.predict(z[None], verbose=0)[0]      # pred[t] ~ z[t+1]
    r = np.zeros_like(z); r[1:] = z[1:] - pred[:-1]
    return (r / rstd).astype(np.float32)

def control_resid_std(pred, idx, mu, sd):
    rs = []
    for i in idx:
        if records[i]["label"] != 0:
            continue
        z = ((records[i]["proc"] - mu) / sd).astype(np.float32)
        p = pred.predict(z[None], verbose=0)[0]
        r = np.zeros_like(z); r[1:] = z[1:] - p[:-1]
        rs.append(r)
    return np.std(np.concatenate(rs), axis=0) + 1e-8

def prepare_fold(train_idx, eval_idx, variant, cfg, seed=SEED):
    """Fit fold-local features on train_idx controls, transform train+eval.
    Writes records[i]['feat'] for every i in train_idx + eval_idx."""
    if variant in ("baseline", "A"):
        C, stds = fit_lp([records[i]["proc"] for i in train_idx
                          if records[i]["label"] == 0])
        set_feats(train_idx + eval_idx, C, stds)
    elif variant == "B":
        pred, mu, sd = train_predictor(train_idx, hidden=cfg["gru_hidden"], seed=seed)
        rstd = control_resid_std(pred, train_idx, mu, sd)
        for i in train_idx + eval_idx:
            records[i]["feat"] = predictor_residual(pred, records[i]["proc"],
                                                    mu, sd, rstd)

## Hyperparameter search (random search, nested-lite)

For each variant we sample `N_TRIALS` random configs and score them by **mean
AUC over an inner 3-fold subject-level CV inside the outer fold-0 training
split** — LP / GRU predictor / normalisation are always refit inside the inner
train split, so the search itself is leakage-free.

> **Selection-bias caveat:** because the search uses the outer fold-0 split,
> the final 10-fold numbers for A and B are mildly optimistic. Treat them as
> exploratory; a fully nested search (per outer fold) costs ~10× more compute —
> set `SEARCH_FOLD` per fold if you want it.

In [ ]:
# ============================================================
# Random search spaces (kept close to the paper's scale: ~4-6k params)
# ============================================================
SPACE_A = dict(channels=[32, 48], kernels=[(7, 5, 3), (9, 7, 5)], se_ratio=[4, 8],
               spat_drop=[0.1, 0.2, 0.3], drop=[0.25, 0.4],
               lr=[5e-4, 1e-3], l2=[1e-4, 1e-3])
SPACE_B = dict(gru_hidden=[16, 32], channels=[32, 48], spat_drop=[0.1, 0.2],
               drop=[0.25, 0.4], lr=[5e-4, 1e-3], l2=[1e-4, 1e-3])

def sample_cfg(space, rng):
    return {k: v[int(rng.integers(len(v)))] for k, v in space.items()}

def inner_cv_score(variant, cfg, outer_train_idx, n_inner=N_INNER, seed=SEED):
    """Mean AUC over inner subject-level CV; fold-local fits inside each split."""
    subjs = sorted({records[i]["subject"] for i in outer_train_idx})
    ys = [subj_label[s] for s in subjs]
    skf_in = StratifiedKFold(n_inner, shuffle=True, random_state=seed)
    aucs = []
    for j, (tr, va) in enumerate(skf_in.split(np.zeros(len(subjs)), ys)):
        tr_subj = {subjs[i] for i in tr}; va_subj = {subjs[i] for i in va}
        tr_i = [i for i in outer_train_idx if records[i]["subject"] in tr_subj]
        va_i = [i for i in outer_train_idx if records[i]["subject"] in va_subj]
        prepare_fold(tr_i, va_i, variant, cfg, seed=seed + 10 * j)
        model = train_window_stage(builder_for(variant, cfg), tr_i,
                                   seed=seed + j, lr=cfg.get("lr", LR_WIN))
        clf = recalibrate_fc(model, tr_i)
        p = predict_recordings(model, clf, va_i)
        aucs.append(roc_auc_score([records[i]["label"] for i in va_i], p))
    return float(np.mean(aucs))

def random_search(variant, space, n_trials=N_TRIALS, seed=SEED):
    rng = np.random.default_rng(seed)
    train_idx = FOLDS[SEARCH_FOLD]["train"]
    rows, best = [], (None, -np.inf)
    for t in range(n_trials):
        cfg = sample_cfg(space, rng)
        t0 = time.time()
        score = inner_cv_score(variant, cfg, train_idx, seed=seed + 1000 + t)
        rows.append(dict(trial=t, inner_AUC=round(score, 4),
                         **{k: str(v) for k, v in cfg.items()}))
        print(f"[{variant}] trial {t:2d}: inner-CV AUC {score:.3f} "
              f"({time.time()-t0:.0f}s)  {cfg}")
        if score > best[1]:
            best = (cfg, score)
    print(f"\n[{variant}] BEST inner-CV AUC {best[1]:.3f}: {best[0]}")
    return best[0], pd.DataFrame(rows)

In [ ]:
bestA_cfg, searchA = random_search("A", SPACE_A)
bestB_cfg, searchB = random_search("B", SPACE_B)
display(searchA); display(searchB)
print("best A:", bestA_cfg)
print("best B:", bestB_cfg)

## Final evaluation on the 10 outer folds

Best config of each variant, plus the fold-local LPGNet baseline for reference
(same folds, same seeds). Baseline numbers should match Exp 2 of the
replication notebook up to run-to-run noise.

In [ ]:
def run_outer(variant, cfg, label):
    rows, t0 = [], time.time()
    for k, f in enumerate(FOLDS):
        tk = time.time()
        prepare_fold(f["train"], f["test"], variant, cfg, seed=SEED + k)
        model = train_window_stage(builder_for(variant, cfg), f["train"],
                                   seed=SEED + k, lr=cfg.get("lr", LR_WIN))
        clf = recalibrate_fc(model, f["train"])
        p = predict_recordings(model, clf, f["test"])
        m = compute_metrics([records[i]["label"] for i in f["test"]], p)
        rows.append(dict(fold=k, **m))
        print(f"[{label}] fold {k}: AUC {m['AUC']:.3f}  ACC {m['ACC']:.3f}  "
              f"F1 {m['F1']:.3f}  SENS {m['SENS']:.3f}  SPEC {m['SPEC']:.3f} "
              f"({time.time()-tk:.0f}s)")
    df = pd.DataFrame(rows)
    print(f"[{label}] mean±std:", {m: f"{df[m].mean():.3f}±{df[m].std():.3f}"
          for m in ("AUC", "ACC", "F1", "SENS", "SPEC")},
          f"| total {(time.time()-t0)/60:.1f} min")
    return df

res_base = run_outer("baseline", {}, "baseline fold-local LPGNet")
res_A    = run_outer("A", bestA_cfg, "Variant A (best)")
res_B    = run_outer("B", bestB_cfg, "Variant B (best)")

# --- parameter counts (compare with the paper's 4,933) ---
for nm, m in [("LPGNet baseline", build_lpgnet(WIN)),
              ("Variant A best", build_variantA(WIN, **_build_kw(bestA_cfg, "lr"))),
              ("Variant B classifier", build_lpgnet(WIN, **_build_kw(bestB_cfg, "lr", "gru_hidden"))),
              ("Variant B GRU predictor", build_predictor(hidden=bestB_cfg["gru_hidden"]))]:
    print(f"{nm}: {m.count_params():,} trainable params")

## Ablation — input type × inference mode

Fixed LPGNet architecture, fold-local protocol, all 10 folds:

- **input**: raw VGRF (fold-local z-score) · LP residual (fold-local, order 11) · GRU residual (fold-local)
- **inference**: whole-recording (step-2 FC recalibration) · window-averaged (mean of window probabilities)

This isolates how much of the performance comes from the *residual
representation* vs the *inference scheme*.

In [ ]:
abl_rows, t0 = [], time.time()
for k, f in enumerate(FOLDS):
    tk = time.time()
    tr, te = f["train"], f["test"]
    # fold-local statistics for each input type
    C, stds = fit_lp([records[i]["proc"] for i in tr if records[i]["label"] == 0])
    set_feats(tr + te, C, stds, key="feat_lp")
    cat = np.concatenate([records[i]["proc"] for i in tr if records[i]["label"] == 0])
    mu_c, sd_c = cat.mean(0), cat.std(0) + 1e-8
    for i in tr + te:
        records[i]["feat_raw"] = ((records[i]["proc"] - mu_c) / sd_c).astype(np.float32)
    pred, mu, sd = train_predictor(tr, hidden=32, seed=SEED + k)
    rstd = control_resid_std(pred, tr, mu, sd)
    for i in tr + te:
        records[i]["feat_gru"] = predictor_residual(pred, records[i]["proc"], mu, sd, rstd)
    for key, nm in [("feat_raw", "raw VGRF"), ("feat_lp", "LP residual"),
                    ("feat_gru", "GRU residual")]:
        model = train_window_stage(build_lpgnet, tr, seed=SEED + k, key=key)
        clf = recalibrate_fc(model, tr, key=key)
        for mode, p in [("whole-recording", predict_recordings(model, clf, te, key=key)),
                        ("window-averaged", predict_window_avg(model, te, key=key))]:
            m = compute_metrics([records[i]["label"] for i in te], p)
            abl_rows.append(dict(fold=k, input=nm, inference=mode, **m))
    print(f"fold {k} ablation done ({time.time()-tk:.0f}s)")

abl = pd.DataFrame(abl_rows)
print(f"total ablation time: {(time.time()-t0)/60:.1f} min")
piv = abl.groupby(["input", "inference"])[["AUC", "ACC", "F1", "SENS", "SPEC"]]
piv = piv.agg(lambda s: f"{s.mean():.3f}±{s.std():.3f}")
display(piv)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, m in zip(axes, ("AUC", "F1")):
    sns.boxplot(data=abl, x="input", y=m, hue="inference", ax=ax)
    ax.set_ylim(0.4, 1.02); ax.set_title(f"{m} by input type and inference mode")
plt.tight_layout(); plt.show()

## Grand summary

In [ ]:
# ============================================================
# GRAND SUMMARY
# ============================================================
def _ms(df, m): return f"{df[m].mean():.3f}±{df[m].std():.3f}"

summary = pd.DataFrame({
    "config": ["LPGNet fold-local (baseline)",
               f"Variant A best {bestA_cfg}",
               f"Variant B best {bestB_cfg}"],
    "AUC":  [_ms(res_base, "AUC"), _ms(res_A, "AUC"), _ms(res_B, "AUC")],
    "ACC":  [_ms(res_base, "ACC"), _ms(res_A, "ACC"), _ms(res_B, "ACC")],
    "F1":   [_ms(res_base, "F1"),  _ms(res_A, "F1"),  _ms(res_B, "F1")],
    "SENS": [_ms(res_base, "SENS"), _ms(res_A, "SENS"), _ms(res_B, "SENS")],
    "SPEC": [_ms(res_base, "SPEC"), _ms(res_A, "SPEC"), _ms(res_B, "SPEC")],
})
display(summary)

print("paired Wilcoxon vs baseline (10 matched folds):")
for nm, df in [("Variant A", res_A), ("Variant B", res_B)]:
    for m in ("AUC", "ACC", "F1"):
        if (res_base[m] != df[m]).any():
            print(f"  {nm} {m}: p={wilcoxon(res_base[m], df[m]).pvalue:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
box = pd.DataFrame({"baseline": res_base["AUC"], "Variant A": res_A["AUC"],
                    "Variant B": res_B["AUC"]})
sns.boxplot(data=box, ax=ax, width=0.5)
sns.stripplot(data=box, ax=ax, color="k", size=5, alpha=0.6)
ax.set_ylim(0.4, 1.02); ax.set_title("AUC across the 10 outer folds")
plt.tight_layout(); plt.show()

## Optional: Mamba predictor (experimental)

In [ ]:
# ============================================================
# OPTIONAL: Mamba next-step predictor (EXPERIMENTAL, skipped by default)
# mamba-ssm needs a CUDA build; on Colab GPU it usually installs, but if it
# fails this cell degrades gracefully. Flip ENABLE_MAMBA to True to try.
# ============================================================
ENABLE_MAMBA = False

if ENABLE_MAMBA:
    try:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "mamba-ssm", "causal-conv1d"], check=True)
        import torch, torch.nn as nn
        from mamba_ssm import Mamba

        class MambaPredictor(nn.Module):
            def __init__(self, n_ch=18, d_model=32):
                super().__init__()
                self.inp = nn.Linear(n_ch, d_model)
                self.mamba = Mamba(d_model=d_model, d_state=16, d_conv=4, expand=2)
                self.out = nn.Linear(d_model, n_ch)
            def forward(self, x):
                return self.out(self.mamba(self.inp(x)))

        # quick demo: train on outer fold-0 train controls, report val MSE
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        ctr = [i for i in FOLDS[0]["train"] if records[i]["label"] == 0]
        tr_i, va_i = split_val(ctr, frac=0.15, seed=SEED)
        cat = np.concatenate([records[i]["proc"] for i in tr_i])
        mu, sd = cat.mean(0), cat.std(0) + 1e-8
        Xtr, Ytr = pred_segments(tr_i, mu, sd); Xva, Yva = pred_segments(va_i, mu, sd)
        net = MambaPredictor().to(dev)
        opt = torch.optim.Adam(net.parameters(), lr=1e-3)
        lossf = nn.MSELoss()
        Xtr_t = torch.tensor(Xtr, device=dev); Ytr_t = torch.tensor(Ytr, device=dev)
        Xva_t = torch.tensor(Xva, device=dev); Yva_t = torch.tensor(Yva, device=dev)
        for ep in range(15):
            net.train(); perm = torch.randperm(len(Xtr_t), device=dev); tot = 0
            for b in range(0, len(perm), 32):
                bi = perm[b:b+32]
                opt.zero_grad()
                loss = lossf(net(Xtr_t[bi]), Ytr_t[bi])
                loss.backward(); opt.step(); tot += loss.item() * len(bi)
            net.eval()
            with torch.no_grad():
                vl = lossf(net(Xva_t), Yva_t).item()
            print(f"epoch {ep}: train MSE {tot/len(perm):.4f}  val MSE {vl:.4f}")
        n_params = sum(p.numel() for p in net.parameters())
        print(f"Mamba predictor params: {n_params:,}")
        print("To use it as a Variant-B predictor, port the residual extraction"
              " from predictor_residual() to this model.")
    except Exception as e:
        print("Mamba unavailable in this runtime — skipping. Reason:", e)
else:
    print("Mamba cell skipped (ENABLE_MAMBA=False).")

## Notes & caveats

- **Search budget** (`N_TRIALS = 12`, inner 3-fold CV) is a compute/quality
  compromise. For a publication-grade result, raise `N_TRIALS` and consider a
  fully nested search (`SEARCH_FOLD` per outer fold, ~10× compute).
- **Selection bias:** the final A/B numbers are mildly optimistic because the
  search saw the outer fold-0 split. The baseline is unaffected (no search).
- **Variant B** replaces the handcrafted order-11 LP with a learned nonlinear
  predictor (GRU, 16–32 hidden units — same order of magnitude of parameters as
  the 198 LP coefficients). It is trained per fold on fold-local controls only,
  so it is the leakage-free analogue of the paper's LP step, and directly tests
  whether a *learned* predictor residual beats the linear one.
- **Variant A** keeps the LP residual input but upgrades the classifier with
  squeeze-excitation (channel attention) and attention pooling instead of plain
  global average pooling, staying in the same ~4–6k parameter regime.
- The **ablation** separates representation (input type) from inference scheme;
  expect whole-recording inference to dominate window-averaging for residual
  inputs, per the paper's step-2 design.
- All runs use fixed seeds, but GPU nondeterminism means small run-to-run
  variation is expected.
